# LSTM Autoencoder — 공정 시뮬레이터 시계열 이상 탐지 (6주차 확장, 선택 항목)

SECOM(`03_modeling.ipynb`)의 Isolation Forest/XGBoost는 정적 피처 벡터를 다룬다. 이 노트북은 로드맵에 남겨뒀던 선택 항목 — **시뮬레이터가 만드는 시계열 공정 데이터**에 대해 LSTM Autoencoder로 재구성 오류 기반 이상 탐지를 시도한다.

**SECOM 모델과는 완전히 별개의 세 번째 탐지기**다: 입력 공간(공정 파라미터 시계열)도, 데이터 출처(시뮬레이터)도, 학습 방식(비지도 재구성)도 다르다. `backend/`/`frontend/`에는 아직 연결하지 않았다 — 로드맵상 이 항목은 "시간 여유가 있을 때 추가"하는 노트북 수준 실험으로 남겨뒀던 것이라, 여기서도 실험/평가까지만 다루고 API 연동은 다음 단계로 남긴다.

**방법**
1. `ProcessSimulator`로 공정별 시계열(연속 판독값) 생성
2. 길이 T 슬라이딩 윈도우로 자르고, 윈도우의 라벨은 '마지막 시점이 이상인가'
3. 정상 윈도우만으로 LSTM Autoencoder 학습 (비지도 — 정상 패턴 재구성을 배운다)
4. 임계값은 **학습에 쓴 정상 윈도우의 재구성 오류 분포**에서 정함 (test 라벨을 보지 않음)
5. 마지막에 test(정상+이상 혼합)로 F1/AUROC 평가

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, precision_recall_curve

from simulator import PROCESS_SPECS, ProcessSimulator

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
WINDOW = 10
MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

device = torch.device("cpu")
print("processes:", list(PROCESS_SPECS))

processes: ['wafer_fabrication', 'oxidation', 'photolithography', 'etching', 'deposition', 'metallization', 'eds', 'packaging']


## 1. 시계열 생성 + 윈도우화

시뮬레이터는 샘플 간 시간적 상관관계를 모델링하지 않으므로(각 호출이 독립), 여기서는 생성 순서를 곧 시간 축으로 취급하는 합성 시계열이라는 점을 명시한다. 정규화는 공정 스펙의 [low, high] 범위를 이용한 min-max 스케일링(0~1)을 쓴다 — 이상치는 이 범위를 벗어나므로 스케일링 후에도 0~1 밖의 값으로 남아 신호가 유지된다.

In [2]:
def make_windows(values, labels, window):
    X, y = [], []
    for i in range(window, len(values) + 1):
        X.append(values[i - window:i])
        y.append(labels[i - 1])  # 마지막 시점의 라벨
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


def build_dataset(process, n_samples=4000, anomaly_ratio=0.1, seed=RANDOM_STATE):
    sim = ProcessSimulator(seed=seed)
    df = sim.generate_batch(process, n_samples=n_samples, anomaly_ratio=anomaly_ratio)
    param_names = [spec.name for spec in PROCESS_SPECS[process]]
    lows = np.array([spec.low for spec in PROCESS_SPECS[process]])
    highs = np.array([spec.high for spec in PROCESS_SPECS[process]])

    raw = df[param_names].to_numpy(dtype=np.float32)
    scaled = (raw - lows) / (highs - lows)  # 정상 범위 -> 대략 [0, 1], 이상치는 범위 밖
    labels = df["is_anomaly"].to_numpy(dtype=np.int64)

    X, y = make_windows(scaled, labels, WINDOW)
    return X, y, param_names

## 2. LSTM Autoencoder 정의

In [3]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, hidden_size=16):
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)
        self.hidden_size = hidden_size

    def forward(self, x):
        seq_len = x.size(1)
        _, (h_n, _) = self.encoder(x)
        latent = h_n[-1]  # (batch, hidden_size) — 시퀀스 전체를 요약한 벡터
        decoder_input = latent.unsqueeze(1).repeat(1, seq_len, 1)
        decoded, _ = self.decoder(decoder_input)
        return self.output_layer(decoded)

## 3. 공정별 학습 + 평가

정상 윈도우만으로 학습(train/val 80:20 분리, val도 정상만 — 조기 종료용), 임계값은 train 재구성 오류의 99th 백분위수로 정한다. test는 정상+이상 혼합, 마지막에 한 번만 평가.

In [4]:
def reconstruction_error(model, X):
    """윈도우 전체 평균 MSE 대신 '마지막 시점'만의 MSE를 쓴다.

    처음엔 시퀀스 전체 평균으로 했더니 F1이 낮았다(평균 ~0.20) — 윈도우 10칸 중
    이상은 마지막 1칸뿐인데 나머지 9칸(정상)이 평균에 섞이면서 신호가 희석된 것.
    라벨이 '마지막 시점이 이상인가'이므로, 점수도 마지막 시점의 재구성 오류만 보는 게 맞다.
    """
    model.eval()
    with torch.no_grad():
        x = torch.from_numpy(X)
        recon = model(x)
        err = ((recon[:, -1, :] - x[:, -1, :]) ** 2).mean(dim=1).numpy()
    return err


def train_one_process(process, epochs=30, lr=1e-3, batch_size=32):
    X, y, param_names = build_dataset(process)
    n_features = len(param_names)

    normal_X = X[y == 0]
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.permutation(len(normal_X))
    split = int(len(idx) * 0.8)
    train_X = normal_X[idx[:split]]
    val_X = normal_X[idx[split:]]

    # test: 별도 시뮬레이션(다른 seed)으로 완전히 분리, 정상+이상 혼합
    test_X, test_y, _ = build_dataset(process, n_samples=1200, anomaly_ratio=0.15, seed=RANDOM_STATE + 1)

    model = LSTMAutoencoder(n_features=n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_tensor = torch.from_numpy(train_X)
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(train_tensor))
        epoch_loss = 0.0
        for i in range(0, len(perm), batch_size):
            batch = train_tensor[perm[i:i + batch_size]]
            optimizer.zero_grad()
            recon = model(batch)
            loss = loss_fn(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)
        epoch_loss /= len(train_tensor)

    train_err = reconstruction_error(model, train_X)
    threshold = float(np.percentile(train_err, 99))

    test_err = reconstruction_error(model, test_X)
    test_pred = (test_err > threshold).astype(int)

    metrics = {
        "process": process,
        "final_train_loss": epoch_loss,
        "threshold": threshold,
        "precision": precision_score(test_y, test_pred, zero_division=0),
        "recall": recall_score(test_y, test_pred, zero_division=0),
        "f1": f1_score(test_y, test_pred, zero_division=0),
        "auroc": roc_auc_score(test_y, test_err),
    }
    return model, metrics

In [5]:
results = []
models_by_process = {}

for process in PROCESS_SPECS:
    model, metrics = train_one_process(process)
    models_by_process[process] = model
    results.append(metrics)
    print(f"{process:>18}: F1={metrics['f1']:.4f}  AUROC={metrics['auroc']:.4f}  "
          f"precision={metrics['precision']:.4f}  recall={metrics['recall']:.4f}")

results_df = pd.DataFrame(results)
results_df

 wafer_fabrication: F1=0.9674  AUROC=1.0000  precision=0.9369  recall=1.0000


         oxidation: F1=0.9461  AUROC=1.0000  precision=0.8977  recall=1.0000


  photolithography: F1=0.9578  AUROC=1.0000  precision=0.9190  recall=1.0000


           etching: F1=0.9554  AUROC=1.0000  precision=0.9147  recall=1.0000


        deposition: F1=0.9531  AUROC=1.0000  precision=0.9104  recall=1.0000


     metallization: F1=0.9674  AUROC=1.0000  precision=0.9369  recall=1.0000


               eds: F1=0.9534  AUROC=0.9998  precision=0.9109  recall=1.0000


         packaging: F1=0.9461  AUROC=1.0000  precision=0.8977  recall=1.0000


,process,final_train_loss,threshold,precision,recall,f1,auroc
0,wafer_fabrication,0.066527,0.118995,0.936893,1.0,0.967419,0.999995
1,oxidation,0.063724,0.110885,0.897674,1.0,0.946078,1.000000
2,photolithography,0.064207,0.115860,0.919048,1.0,0.957816,0.999995
3,etching,0.067692,0.117248,0.914692,1.0,0.955446,1.000000
4,deposition,0.067044,0.103222,0.910377,1.0,0.953086,1.000000
5,metallization,0.062961,0.116281,0.936893,1.0,0.967419,0.999990
6,eds,0.063872,0.126593,0.910891,1.0,0.953368,0.999817
7,packaging,0.069532,0.104683,0.897674,1.0,0.946078,1.000000


## 4. 결과 저장

공정별 LSTM Autoencoder 가중치와 평가 결과를 저장한다.

In [6]:
for process, model in models_by_process.items():
    torch.save(model.state_dict(), MODELS_DIR / f"lstm_ae_{process}.pt")

results_df.to_csv(MODELS_DIR / "lstm_autoencoder_results.csv", index=False)
print("저장 완료:", sorted(p.name for p in MODELS_DIR.glob("lstm_ae_*.pt")))
print(f"평균 F1={results_df['f1'].mean():.4f}, 평균 AUROC={results_df['auroc'].mean():.4f}")

저장 완료: ['lstm_ae_deposition.pt', 'lstm_ae_eds.pt', 'lstm_ae_etching.pt', 'lstm_ae_metallization.pt', 'lstm_ae_oxidation.pt', 'lstm_ae_packaging.pt', 'lstm_ae_photolithography.pt', 'lstm_ae_wafer_fabrication.pt']
평균 F1=0.9558, 평균 AUROC=1.0000


## 결론

**처음 시도(윈도우 전체 평균 재구성 오류)는 성능이 낮았다** — 공정별 F1 0.18~0.22, AUROC 0.70~0.73 수준. 원인을 보니 윈도우 길이 10 중 이상은 마지막 1칸뿐인데, 나머지 9칸(정상)의 낮은 오류가 평균에 섞여 신호가 희석되고 있었다. 라벨 자체가 '마지막 시점이 이상인가'이므로, 점수도 전체 평균이 아니라 **마지막 시점의 재구성 오류만** 보도록 바꿨더니 공정 전체 평균 F1이 0.9 이상으로 뛰었다(아래 실행 결과).

다만 이 높은 점수는 시뮬레이터 데이터의 한계도 함께 봐야 한다: 이상치는 정상 범위를 확실히 벗어나도록 규칙으로 생성되므로(2주차 `simulator/process_simulator.py`), 정상 패턴만 학습한 오토인코더 입장에서는 구분이 SECOM(실측 노이즈 센서)보다 훨씬 쉽다. 그래도 얻은 교훈은 실전에도 적용된다: **시계열 이상 탐지에서 윈도우 집계 방식이 성능을 좌우할 수 있다**는 것 — 어떤 시점의 이상을 탐지하려는 것인지에 맞춰 점수를 설계해야 한다.

**다음 단계 (연동하려면)**: `backend/app/lstm_ml.py`를 만들어 공정별 모델을 로드하고, `/api/process/history`에서 최근 `WINDOW`개 판독값을 모아 마지막 시점 재구성 오류를 계산하는 새 엔드포인트(예: `/api/ai/detect-timeseries`)를 추가하면 대시보드에도 붙일 수 있다. 이번 실험에서는 로드맵 원안대로 노트북 단계까지만 진행했다.